# Datathon 2026 — RADİKAL (A): LLM ile yapılandırılmış metin çıkarımı

Qwen2.5-7B-Instruct ile her mentor metninden AÇIK SEMANTİK feature'lar: övülen/eleştirilen
beceriler (14'er bayrak), mentor tonu (1-10), gelişim beklentisi (1-10), somut başarı (0/1).
TF-IDF/BERT'in vektörde kaybettiği ayrık sinyal -> GBM feature'ı olur. Hedef KULLANILMAZ (sızıntısız).

## KURULUM (sırayla):
1. **Add Input → Models** → ara: **"Qwen2.5 7B Instruct"** (qwen-lm / Transformers formatı) → Add
   (bulamazsan alternatif: "gemma-2 9b it" — kod her ikisini de otomatik bulur)
2. **Add Input → Competitions → Datathon 2026**
3. **Settings → Accelerator → GPU T4 ×2** → Restart  (Internet GEREKMEZ, model lokal)
4. Run All

## Süre: ~2-3 saat (20k metin, batch'li). Her 1000 satırda ara kayıt -> kesilirse kaldığı yerden devam.
## Çıktı: `llm_feats_train.csv` + `llm_feats_test.csv` -> indir, bana yükle.

In [ ]:
# ===================== LLM yapılandırılmış çıkarım — TEK HÜCRE =====================
import os, glob, json, re, time
import numpy as np, pandas as pd, torch
assert torch.cuda.is_available(), "GPU YOK! Settings->Accelerator->GPU T4 x2 -> Restart"
print("GPU:", torch.cuda.get_device_name(0), "x", torch.cuda.device_count())

# ---------- model yolu (Qwen veya Gemma otomatik) ----------
cands = [os.path.dirname(p) for p in glob.glob('/kaggle/input/**/config.json', recursive=True)
         if any(k in p.lower() for k in ('qwen','gemma'))]
assert cands, "Model yok! Add Input -> Models -> 'Qwen2.5 7B Instruct' ekle"
MODEL_PATH = sorted(cands, key=len)[0]
print("MODEL:", MODEL_PATH)

from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained(MODEL_PATH, padding_side='left')
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, torch_dtype=torch.float16, device_map='auto')
model.eval()

# ---------- veri ----------
base = [p for p in glob.glob('/kaggle/input/*') if os.path.exists(f'{p}/train.csv')]
if not base:
    hits = glob.glob('/kaggle/input/**/train.csv', recursive=True); base=[os.path.dirname(hits[0])]
base = base[0]
train = pd.read_csv(f'{base}/train.csv'); test = pd.read_csv(f'{base}/test_x.csv')
TEXT='mentor_feedback_text'; ID='student_id'

SKILLS = ['kodlama','problem_cozme','veri_yapilari','sql','ml','backend','frontend',
          'cloud','devops','iletisim','takim','liderlik','sunum','proje_kalitesi']
SYS = ("Mentor değerlendirme metnini analiz et. SADECE şu JSON'u döndür, başka hiçbir şey yazma:\n"
       '{"ovulen":[...],"elestirilen":[...],"ton":N,"gelisim":N,"somut_basari":0veya1}\n'
       f"ovulen/elestirilen: şu listeden uygun olanlar: {SKILLS}\n"
       "ton: metnin genel olumluluk tonu 1-10. gelisim: mentorun gelişim beklentisi 1-10.\n"
       "somut_basari: somut bir başarı/ödül/proje bahsi varsa 1 yoksa 0.")

def make_prompts(texts):
    msgs = [[{"role":"system","content":SYS},{"role":"user","content":t[:1200]}] for t in texts]
    return [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in msgs]

def parse_one(s):
    m = re.search(r'\{.*\}', s, re.DOTALL)
    out = {'ton':5,'gelisim':5,'somut_basari':0,'parse_ok':0}
    for sk in SKILLS: out[f'ov_{sk}']=0; out[f'el_{sk}']=0
    if not m: return out
    try:
        d = json.loads(m.group(0))
        for sk in d.get('ovulen',[]) or []:
            k=str(sk).strip().lower().replace(' ','_')
            if k in SKILLS: out[f'ov_{k}']=1
        for sk in d.get('elestirilen',[]) or []:
            k=str(sk).strip().lower().replace(' ','_')
            if k in SKILLS: out[f'el_{k}']=1
        out['ton']=int(np.clip(float(d.get('ton',5)),1,10))
        out['gelisim']=int(np.clip(float(d.get('gelisim',5)),1,10))
        out['somut_basari']=1 if d.get('somut_basari') in (1,'1',True) else 0
        out['parse_ok']=1
    except Exception: pass
    return out

@torch.no_grad()
def extract(texts, tag, bs=16):
    part = f'/kaggle/working/llm_partial_{tag}.csv'
    done = 0; rows=[]
    if os.path.exists(part):
        prev = pd.read_csv(part); rows = prev.to_dict('records'); done = len(rows)
        print(f"[{tag}] devam: {done} hazır")
    t0=time.time()
    for s in range(done, len(texts), bs):
        batch = texts[s:s+bs]
        enc = tok(make_prompts(batch), return_tensors='pt', padding=True, truncation=True, max_length=512).to(model.device)
        out = model.generate(**enc, max_new_tokens=96, do_sample=False, pad_token_id=tok.pad_token_id)
        dec = tok.batch_decode(out[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)
        rows += [parse_one(d) for d in dec]
        if (s//bs) % 8 == 0:
            el=time.time()-t0; done_n=len(rows)-done
            eta=(len(texts)-len(rows))/max(done_n/el,1e-9)/60 if done_n else -1
            print(f"[{tag}] {len(rows)}/{len(texts)}  ({el/60:.1f}dk, ETA {eta:.0f}dk)", flush=True)
        if (s//bs) % 16 == 0 and len(rows)>done:
            pd.DataFrame(rows).to_csv(part, index=False)   # ara kayıt
    df = pd.DataFrame(rows); df.to_csv(part, index=False)
    return df

for tag, frame in [('train', train), ('test', test)]:
    feats = extract(frame[TEXT].fillna('').tolist(), tag)
    feats.insert(0, ID, frame[ID].values)
    feats.to_csv(f'/kaggle/working/llm_feats_{tag}.csv', index=False)
    print(f"[{tag}] BİTTİ -> llm_feats_{tag}.csv  | parse_ok oranı: {feats['parse_ok'].mean():.3f}")

print("\nHEPSİ BİTTİ — llm_feats_train.csv + llm_feats_test.csv indir, bana yükle.")
